# RSNA Knee — fine-tuned ResNet18 (3 planes, attention pooling over slices)
GPU必須。Internet ONでImageNet重みを取得。画像は224px×16スライスをディスクにキャッシュし、エンドツーエンドで学習する。

In [ ]:
import os, sys, subprocess, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
SRC = Path('/kaggle/working/repo/rsna-knee-abnormality-detection/src')
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb','/kaggle/working/repo'], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS
from multilingual_labels import add_pseudo_labels_ml
import finetune as ft, torch
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
train = pd.read_csv(COMP/'train.csv'); tseries = pd.read_csv(COMP/'train_series.csv')
test = pd.read_csv(COMP/'test.csv');   teseries = pd.read_csv(COMP/'test_series.csv')
print(train.shape, tseries.shape, test.shape)
train['has_true'] = train['ACL'].notna()   # 正解ラベルあり(疑似ラベルではない)
print('true-labeled:', int(train.has_true.sum()))
train = add_pseudo_labels_ml(train)        # 英語+トルコ語+ギリシャ語+ブルガリア語+スペイン語+ドイツ語+フランス語のキーワード規則
print(train.lang.value_counts().to_dict())
# 言語ごとの疑似陽性率(本文は出さない)。英語と桁が大きく違う言語は規則が弱い疑い
print(train[~train.has_true].groupby('lang')[LABELS].mean().round(3).T)
# 診断: 規則の弱い言語(bg, tr)と未対応('other')の頻出語。本文は出さず、語の文書頻度だけを出す
import re
from collections import Counter
def top_words(mask, n):
    docs = train.loc[mask, 'Report'].fillna('').astype(str).str.lower()
    c = Counter(w for t in docs for w in set(re.findall(r'[^\W\d_]{4,}', t)))
    return c.most_common(n)
for lg, n in (('bg', 120), ('tr', 120), ('other', 150)):
    print(f'top words [{lg}] ({int((train.lang == lg).sum())} reports):', top_words(train.lang == lg, n))


In [ ]:
SIZE, K, EPOCHS, BS = 224, 16, 10, 8
CACHE = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')   # /kaggle/working に置くと出力に含まれてしまう
USE_LANGS = ['en', 'tr', 'el', 'bg', 'es', 'de', 'fr']   # 使う言語。英語のみで比較したいときは ['en']
use = train[train.has_true | train.lang.isin(USE_LANGS)].reset_index(drop=True)   # 疑似ラベル + 正解ラベル付き
print(use.lang.value_counts().to_dict())
print('studies:', len(use), '| true-labeled:', int(use.has_true.sum()))
X, M = ft.cache_volumes(use, tseries, COMP/'train_series', SIZE, K, CACHE, 'train')
print('planes available (Sag/Cor/Ax):', M.sum(0))
Y = use[LABELS].values.astype(np.float32)

In [ ]:
N_SEEDS = 3                                   # 乱数と検証分割を変えて3モデル。推論時に予測を平均する
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
OUT = Path('model_out'); OUT.mkdir(exist_ok=True)
true_idx = np.where(use.has_true.values)[0]                       # 58件は評価専用(正解ラベル)
pseudo_all = np.where(~use.has_true.values)[0]
nets, true_preds = [], []
for seed in range(N_SEEDS):
    rng = np.random.RandomState(seed); pseudo = pseudo_all.copy(); rng.shuffle(pseudo)
    n_va = int(0.1 * len(pseudo))
    va_pseudo, tr_idx = pseudo[:n_va], pseudo[n_va:]                # シードごとに別の10%を検証に使う
    torch.manual_seed(seed); np.random.seed(seed)
    print(f'=== seed {seed}: train {len(tr_idx)} val-pseudo {len(va_pseudo)}')
    net, hist = ft.fit(X, M, Y, tr_idx, {'pseudo': va_pseudo, 'true': true_idx}, dev, epochs=EPOCHS, bs=BS)
    print('best:', max(hist, key=lambda h: h.get('pseudo', -1)))
    torch.save({'state': net.state_dict(), 'size': SIZE, 'k': K, 'labels': LABELS}, OUT/f'ft_model_s{seed}.pt')  # 各モデルをすぐ保存
    true_preds.append(ft.predict(net, X, M, true_idx, dev))
    nets.append(net.cpu()); torch.cuda.empty_cache()

In [ ]:
for i, P in enumerate(true_preds):
    print(f'seed {i}: macro AUC vs TRUE labels {ft.macro_auc(Y[true_idx], P):.4f}')
print(f'ensemble ({N_SEEDS} models): macro AUC vs TRUE labels {ft.macro_auc(Y[true_idx], np.mean(true_preds, axis=0)):.4f}')